# 第 12 课｜中心极限定理在金融

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`12_youtube-s3Kbk9TCZfI/index.html`  
> **所属模块**：P1-M3 · 统计证据与可预测性

## 本节量化问题

样本均值何时接近正态，为什么这不意味着单期收益正态？

## 准确定义与策略报告位置

在独立同分布且方差有限等条件下，标准化样本均值随样本量增加趋近正态。

- **核心关系**：√n(X̄-μ)/σ 在经典条件下依分布收敛到 N(0,1)。
- **写入报告**：估计误差、置信区间和有效样本量
- **P2 衔接**：P2 第 48、53 课：统计工具与并行模拟。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 观察样本均值分布的收敛

底层收益使用偏斜分布；对不同样本量重复抽样，比较样本均值的标准化偏度和峰度。

In [2]:
from scipy import stats

rng = np.random.default_rng(365)
sample_sizes = [5, 20, 100]
rows = []

for n in sample_sizes:
    samples = rng.lognormal(mean=0.0, sigma=0.7, size=(5_000, n))
    means = samples.mean(axis=1)
    standardized = (means - means.mean()) / means.std(ddof=1)
    rows.append({
        "sample_size": n,
        "skew": stats.skew(standardized, bias=False),
        "excess_kurtosis": stats.kurtosis(standardized, fisher=True, bias=False),
        "std_of_mean": means.std(ddof=1),
    })

display(pd.DataFrame(rows))

,sample_size,skew,excess_kurtosis,std_of_mean
0,5,1.226085,2.910818,0.452627
1,20,0.669121,0.863329,0.228632
2,100,0.301143,0.162272,0.101915


## 方法边界与阶段验收

金融序列的自相关、异方差和厚尾会降低有效样本量或破坏经典条件。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
